In [10]:
GRAPH_VARIANT = "ast_only"

In [11]:
from pathlib import Path

CONTENT_ROOT = Path("/content")
VARIANT_UPLOAD_DIR = CONTENT_ROOT / GRAPH_VARIANT

EMBEDDING_ZIP_CANDIDATES = list(CONTENT_ROOT.glob("shared_token_embedding_*.zip"))
assert EMBEDDING_ZIP_CANDIDATES, "Không tìm thấy shared_token_embedding_*.zip trong /content"

EMBEDDING_ZIP = EMBEDDING_ZIP_CANDIDATES[0]
EMBEDDING_ROOT = CONTENT_ROOT / "embedding" / EMBEDDING_ZIP.stem

GRAPH_ROOT = CONTENT_ROOT / "graphs"
OUTPUT_ROOT = CONTENT_ROOT / "training_output"

print("GRAPH_VARIANT:", GRAPH_VARIANT)
print("VARIANT_UPLOAD_DIR:", VARIANT_UPLOAD_DIR, VARIANT_UPLOAD_DIR.exists())
print("EMBEDDING_ZIP:", EMBEDDING_ZIP)
print("EMBEDDING_ROOT:", EMBEDDING_ROOT)
print("GRAPH_ROOT:", GRAPH_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


GRAPH_VARIANT: ast_only
VARIANT_UPLOAD_DIR: /content/ast_only True
EMBEDDING_ZIP: /content/shared_token_embedding_300000.zip
EMBEDDING_ROOT: /content/embedding/shared_token_embedding_300000
GRAPH_ROOT: /content/graphs
OUTPUT_ROOT: /content/training_output


In [12]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/HuyBui2112/detecting_variable_misuse_bugs.git"
BRANCH = "main"
REPO_DIR = Path("/content/detecting_variable_misuse_bugs")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

!git log -1 --oneline
!git status --short


79a7aef (HEAD -> main, origin/main, origin/HEAD) update training


In [13]:
!python --version
!python -m pip --version

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda:", torch.version.cuda)

!nvidia-smi || true
!df -h /content


Python 3.12.13
pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
torch: 2.10.0+cu128
cuda available: True
cuda: 12.8
Fri May 15 12:59:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |

In [ ]:
import zipfile
import shutil

EMBEDDING_ROOT.parent.mkdir(parents=True, exist_ok=True)

if EMBEDDING_ROOT.exists():
    print("Embedding đã tồn tại:", EMBEDDING_ROOT)
else:
    EMBEDDING_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(EMBEDDING_ZIP) as archive:
        archive.extractall(EMBEDDING_ROOT)
    print("Đã giải nén embedding:", EMBEDDING_ROOT)

!find "{EMBEDDING_ROOT}" -maxdepth 1 -type f -printf "%f %s bytes\n"

assert (EMBEDDING_ROOT / "token_to_id.json").exists()
assert (EMBEDDING_ROOT / "embedding_init.pt").exists()


Embedding đã tồn tại: /content/embedding/shared_token_embedding_300000
embedding_init.pt 25602138 bytes
token_to_id.json 1135411 bytes
embedding_config.json 479 bytes
id_to_token.json 1235411 bytes
vocabulary_summary.json 4138 bytes


In [15]:
import zipfile
import shutil
from pathlib import Path

target_variant_root = GRAPH_ROOT / GRAPH_VARIANT
target_train_dir = target_variant_root / "train"
target_dev_dir = target_variant_root / "dev"
target_eval_dir = target_variant_root / "eval"

GRAPH_ROOT.mkdir(parents=True, exist_ok=True)
target_variant_root.mkdir(parents=True, exist_ok=True)

print("Upload dir files:")
for path in sorted(VARIANT_UPLOAD_DIR.glob("*"))[:20]:
    print(path.name)

# 1. Giải nén mọi file zip nếu có.
zip_files = sorted(VARIANT_UPLOAD_DIR.glob("*.zip"))
if zip_files:
    print("Found zip files:", [p.name for p in zip_files])
    for zip_path in zip_files:
        print("Extract:", zip_path, "->", target_variant_root)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(target_variant_root)

# 2. Nếu upload dir đã có train/dev/eval thì copy vào target.
for split in ["train", "dev", "eval"]:
    src_dir = VARIANT_UPLOAD_DIR / split
    dst_dir = target_variant_root / split
    if src_dir.exists() and not dst_dir.exists():
        print("Copy dir:", src_dir, "->", dst_dir)
        shutil.copytree(src_dir, dst_dir)

# 3. Nếu sau giải nén có nested variant folder thì kéo ra.
nested_variant = target_variant_root / GRAPH_VARIANT
if nested_variant.exists():
    for split in ["train", "dev", "eval"]:
        src_dir = nested_variant / split
        dst_dir = target_variant_root / split
        if src_dir.exists() and not dst_dir.exists():
            print("Move nested split:", src_dir, "->", dst_dir)
            shutil.move(str(src_dir), str(dst_dir))

# 4. Nếu shard nằm trực tiếp trong upload dir, đoán split theo tên file.
for split in ["train", "dev", "eval"]:
    dst_dir = target_variant_root / split
    dst_dir.mkdir(parents=True, exist_ok=True)

direct_jsonl = sorted(VARIANT_UPLOAD_DIR.glob("*.jsonl"))
for src in direct_jsonl:
    name = src.name.lower()
    if "train" in name:
        dst = target_train_dir / src.name
    elif "dev" in name:
        dst = target_dev_dir / src.name
    elif "eval" in name:
        dst = target_eval_dir / src.name
    else:
        continue
    if not dst.exists():
        print("Copy shard:", src, "->", dst)
        shutil.copy2(src, dst)

print("Graph target:")
!find "{target_variant_root}" -maxdepth 3 -type f | head -30

train_files = list(target_train_dir.glob("*.jsonl"))
dev_files = list(target_dev_dir.glob("*.jsonl"))

print("train jsonl:", len(train_files))
print("dev jsonl:", len(dev_files))

assert train_files, f"Không thấy train shards trong {target_train_dir}"
assert dev_files, f"Không thấy dev shards trong {target_dev_dir}"


Upload dir files:
ast_only-20260515T122524Z-3-001.zip
Found zip files: ['ast_only-20260515T122524Z-3-001.zip']
Extract: /content/ast_only/ast_only-20260515T122524Z-3-001.zip -> /content/graphs/ast_only
Graph target:
/content/graphs/ast_only/train/shard_00000.jsonl
/content/graphs/ast_only/train/shard_00002.jsonl
/content/graphs/ast_only/train/shard_00001.jsonl
/content/graphs/ast_only/train/shard_00003.jsonl
/content/graphs/ast_only/ast_only/train/shard_00000.jsonl
/content/graphs/ast_only/ast_only/train/shard_00002.jsonl
/content/graphs/ast_only/ast_only/train/shard_00001.jsonl
/content/graphs/ast_only/ast_only/train/shard_00003.jsonl
/content/graphs/ast_only/ast_only/graph_construction_summary.json
/content/graphs/ast_only/ast_only/dev/shard_00000.jsonl
/content/graphs/ast_only/ast_only/eval/shard_00000.jsonl
/content/graphs/ast_only/ast_only/eval/shard_00001.jsonl
/content/graphs/ast_only/dev/shard_00000.jsonl
/content/graphs/ast_only/eval/shard_00000.jsonl
/content/graphs/ast_only/

In [16]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

!PYTHONPATH=src python scripts/train_gnn.py \
  --graph-root "{GRAPH_ROOT}" \
  --embedding-root "{EMBEDDING_ROOT}" \
  --output-root "{OUTPUT_ROOT}/smoke_test" \
  --graph-variant "{GRAPH_VARIANT}" \
  --epochs 1 \
  --batch-size 4 \
  --hidden-dim 64 \
  --message-passing-steps 2 \
  --max-train-graphs 64 \
  --max-dev-graphs 64 \
  --log-every 10 \
  --early-stopping-patience 3 \
  --monitor-metric combined


train epoch=1 step=10 metrics={'loss': 6.116556787490845, 'localization_correct': 12, 'localization_total': 40, 'localization_accuracy': 0.3, 'repair_correct': 4, 'repair_total': 16, 'repair_accuracy': 0.25}
dev epoch=1 step=10 metrics={'loss': 6.615231657028199, 'localization_correct': 19, 'localization_total': 40, 'localization_accuracy': 0.475, 'repair_correct': 7, 'repair_total': 21, 'repair_accuracy': 0.3333333333333333}
{
  "epoch": 1,
  "train": {
    "loss": 5.866983592510223,
    "localization_correct": 26,
    "localization_total": 64,
    "localization_accuracy": 0.40625,
    "repair_correct": 8,
    "repair_total": 26,
    "repair_accuracy": 0.3076923076923077
  },
  "dev": {
    "loss": 6.71835121512413,
    "localization_correct": 26,
    "localization_total": 64,
    "localization_accuracy": 0.40625,
    "repair_correct": 15,
    "repair_total": 38,
    "repair_accuracy": 0.39473684210526316
  },
  "learning_rate": 0.001
}
{
  "epoch": 1,
  "train": {
    "loss": 5.86698

In [17]:
SMOKE_OUTPUT = OUTPUT_ROOT / "smoke_test" / GRAPH_VARIANT

!find "{SMOKE_OUTPUT}" -maxdepth 1 -type f -printf "%f %s bytes\n"


metrics_history.json 569 bytes
best_model.pt 77640501 bytes
training_summary.json 2148 bytes
last_model.pt 77640501 bytes


In [18]:
!PYTHONPATH=src python scripts/train_gnn.py \
  --graph-root "{GRAPH_ROOT}" \
  --embedding-root "{EMBEDDING_ROOT}" \
  --output-root "{OUTPUT_ROOT}" \
  --graph-variant "{GRAPH_VARIANT}" \
  --epochs 20 \
  --batch-size 8 \
  --hidden-dim 128 \
  --message-passing-steps 4 \
  --log-every 100 \
  --early-stopping-patience 3 \
  --early-stopping-min-delta 0.0001 \
  --monitor-metric combined \
  --lr-scheduler-patience 2 \
  --lr-scheduler-factor 0.5


train epoch=1 step=100 metrics={'loss': 5.059280010461808, 'localization_correct': 394, 'localization_total': 800, 'localization_accuracy': 0.4925, 'repair_correct': 101, 'repair_total': 395, 'repair_accuracy': 0.25569620253164554}
train epoch=1 step=200 metrics={'loss': 4.696504264473915, 'localization_correct': 772, 'localization_total': 1600, 'localization_accuracy': 0.4825, 'repair_correct': 233, 'repair_total': 817, 'repair_accuracy': 0.28518971848225216}
train epoch=1 step=300 metrics={'loss': 4.490904896656672, 'localization_correct': 1165, 'localization_total': 2400, 'localization_accuracy': 0.48541666666666666, 'repair_correct': 389, 'repair_total': 1224, 'repair_accuracy': 0.31781045751633985}
train epoch=1 step=400 metrics={'loss': 4.3399261859059335, 'localization_correct': 1566, 'localization_total': 3200, 'localization_accuracy': 0.489375, 'repair_correct': 558, 'repair_total': 1626, 'repair_accuracy': 0.34317343173431736}
train epoch=1 step=500 metrics={'loss': 4.2529724

In [19]:
!find "{OUTPUT_ROOT}/{GRAPH_VARIANT}" -maxdepth 1 -type f -printf "%f %s bytes\n"

metrics_history.json 2461 bytes
best_model.pt 79862005 bytes
last_model.pt 79862197 bytes


In [20]:
from google.colab import drive
from pathlib import Path
import time
import shutil
import os

drive.mount("/content/drive")

BACKUP_ROOT = Path("/content/drive/MyDrive/variable_misuse_training_backup")
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")

BACKUP_DIR = BACKUP_ROOT / RUN_TAG
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("Backup dir:", BACKUP_DIR)


Mounted at /content/drive
Backup dir: /content/drive/MyDrive/variable_misuse_training_backup/20260515_135106


In [21]:
from pathlib import Path

paths = [
    Path("/content/training_output"),
    Path("/content/training_output/ast_only"),
    Path("/content/training_output/ast_next_token"),
    Path("/content/training_output/ast_next_token_data_flow"),
    Path("/content/graphs"),
    Path("/content/embedding"),
    Path("/content/shared_token_embedding_300000.zip"),
]

for path in paths:
    if path.exists():
        if path.is_file():
            size_gb = path.stat().st_size / (1024 ** 3)
            print(f"[FILE] {path} | {size_gb:.3f} GB")
        else:
            size_gb = sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / (1024 ** 3)
            print(f"[DIR ] {path} | {size_gb:.3f} GB")
    else:
        print(f"[MISS] {path}")


[DIR ] /content/training_output | 0.293 GB
[DIR ] /content/training_output/ast_only | 0.149 GB
[MISS] /content/training_output/ast_next_token
[MISS] /content/training_output/ast_next_token_data_flow
[DIR ] /content/graphs | 2.763 GB
[DIR ] /content/embedding | 0.026 GB
[FILE] /content/shared_token_embedding_300000.zip | 0.023 GB


In [22]:
from pathlib import Path
import shutil

src = Path("/content/training_output")
assert src.exists(), "Không thấy /content/training_output"

archive_base = BACKUP_DIR / "training_output"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(src))

print("Saved:", archive_path)


Saved: /content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/training_output.zip


In [23]:
from pathlib import Path
import shutil

embedding_dir = Path("/content/embedding")
embedding_zip_candidates = list(Path("/content").glob("shared_token_embedding_*.zip"))

if embedding_dir.exists():
    archive_path = shutil.make_archive(str(BACKUP_DIR / "embedding"), "zip", root_dir=str(embedding_dir))
    print("Saved embedding dir:", archive_path)

for zip_path in embedding_zip_candidates:
    dst = BACKUP_DIR / zip_path.name
    shutil.copy2(zip_path, dst)
    print("Saved embedding zip:", dst)


Saved embedding dir: /content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/embedding.zip
Saved embedding zip: /content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/shared_token_embedding_300000.zip


In [24]:
from pathlib import Path
import shutil

graphs_dir = Path("/content/graphs")
assert graphs_dir.exists(), "Không thấy /content/graphs"

archive_path = shutil.make_archive(str(BACKUP_DIR / "graphs"), "zip", root_dir=str(graphs_dir))
print("Saved graphs:", archive_path)


Saved graphs: /content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/graphs.zip


In [25]:
for path in BACKUP_DIR.rglob("*"):
    if path.is_file():
        print(path, round(path.stat().st_size / (1024 ** 2), 2), "MB")


/content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/training_output.zip 165.26 MB
/content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/embedding.zip 23.41 MB
/content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/shared_token_embedding_300000.zip 23.41 MB
/content/drive/MyDrive/variable_misuse_training_backup/20260515_135106/graphs.zip 309.68 MB
